# Professor Research Agent

# Utils

In [1]:
# Production credentials from GCS Interoperability tab
import os
from dotenv import load_dotenv
load_dotenv()

GCS_ACCESS_KEY = os.getenv("GCS_ACCESS_KEY")
GCS_SECRET_KEY = os.getenv("GCS_SECRET_KEY") 
BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")

import boto3
import re
from botocore.exceptions import ClientError
from deepagents.backends.protocol import BackendProtocol, WriteResult, EditResult
from deepagents.backends.utils import FileInfo, GrepMatch

class GCSObjectBackend(BackendProtocol):
    def __init__(self, bucket_name: str, endpoint_url: str = "https://storage.googleapis.com", 
                 aws_access_key_id: str = None, aws_secret_access_key: str = None):
        """
        GCS uses S3-compatible credentials. 
        Get these from GCP Console -> Cloud Storage -> Settings -> Interoperability.
        """
        self.s3 = boto3.resource(
            's3',
            endpoint_url=endpoint_url,
            aws_access_key_id=aws_access_key_id,
            aws_secret_access_key=aws_secret_access_key
        )
        self.bucket = self.s3.Bucket(bucket_name)
        self.bucket_name = bucket_name

    def _normalize_path(self, path: str) -> str:
        # Remove leading slash for S3 keys
        return path.lstrip("/")

    def ls_info(self, path: str) -> list[FileInfo]:
        prefix = self._normalize_path(path)
        if prefix and not prefix.endswith('/'):
            prefix += '/'
            
        results = []
        # Use delimiter to simulate directory behavior
        paginator = self.s3.meta.client.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=self.bucket_name, Prefix=prefix, Delimiter='/'):
            # Add "Folders"
            for obj in page.get('CommonPrefixes', []):
                results.append(FileInfo(path="/" + obj['Prefix'], is_dir=True))
            # Add Files
            for obj in page.get('Contents', []):
                if obj['Key'] != prefix: # Don't list the directory itself
                    results.append(FileInfo(
                        path="/" + obj['Key'], 
                        size=obj['Size'], 
                        modified_at=obj['LastModified'].isoformat()
                    ))
        return sorted(results, key=lambda x: x.path)

    def read(self, file_path: str, offset: int = 0, limit: int = 2000) -> str:
        key = self._normalize_path(file_path)
        try:
            obj = self.bucket.Object(key).get()
            content = obj['Body'].read().decode('utf-8')
            lines = content.splitlines()
            # Apply offset/limit and format with line numbers as per protocol
            subset = lines[offset : offset + limit]
            return "\n".join(f"{i+offset+1}|{line}" for i, line in enumerate(subset))
        except ClientError:
            return f"Error: File '{file_path}' not found."

    def write(self, file_path: str, content: str) -> WriteResult:
        key = self._normalize_path(file_path)
        try:
            # Check if exists to enforce create-only semantics if desired
            self.bucket.Object(key).put(Body=content)
            return WriteResult(path=file_path, files_update=None)
        except Exception as e:
            return WriteResult(error=str(e))

    def edit(self, file_path: str, old_string: str, new_string: str, replace_all: bool = False) -> EditResult:
        key = self._normalize_path(file_path)
        try:
            obj = self.bucket.Object(key).get()
            content = obj['Body'].read().decode('utf-8')
            
            count = content.count(old_string)
            if count == 0:
                return EditResult(error=f"String '{old_string}' not found.")
            if count > 1 and not replace_all:
                return EditResult(error=f"Multiple occurrences of '{old_string}' found. Use replace_all=True.")
            
            new_content = content.replace(old_string, new_string) if replace_all else content.replace(old_string, new_string, 1)
            self.bucket.Object(key).put(Body=new_content)
            
            return EditResult(path=file_path, occurrences=count, files_update=None)
        except Exception as e:
            return EditResult(error=str(e))

    def grep_raw(self, pattern: str, path: str | None = None, glob: str | None = None) -> list[GrepMatch] | str:
        # Simple implementation: List files and scan content
        # Production tip: Use GCS Select or an external index for large buckets
        matches = []
        files_to_scan = self.ls_info(path or "/")
        try:
            regex = re.compile(pattern)
            for f in files_to_scan:
                if not f.is_dir:
                    content = self.bucket.Object(self._normalize_path(f.path)).get()['Body'].read().decode('utf-8')
                    for i, line in enumerate(content.splitlines()):
                        if regex.search(line):
                            matches.append(GrepMatch(path=f.path, line=i+1, text=line))
            return matches
        except re.error as e:
            return f"Invalid regex pattern: {str(e)}"

    def glob_info(self, pattern: str, path: str = "/") -> list[FileInfo]:
        # Basic implementation: list everything and filter with regex
        all_files = self.ls_info(path)
        # Convert glob to regex
        regex_pattern = pattern.replace(".", "\\.").replace("*", ".*").replace("?", ".")
        regex = re.compile(regex_pattern)
        return [f for f in all_files if regex.search(f.path)]


f:\anaconda3\envs\gpu_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Backends

In [5]:
import atexit
import os
from contextlib import ExitStack
from dotenv import load_dotenv
from deepagents.backends import StateBackend,StoreBackend,CompositeBackend
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

load_dotenv()
from utils.utils import GCSObjectBackend
GCS_ACCESS_KEY = os.getenv("GCS_ACCESS_KEY")
GCS_SECRET_KEY = os.getenv("GCS_SECRET_KEY") 
BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
DB_URI = os.getenv("DB_URI")
from langgraph.store.postgres import PostgresStore
import os

if DB_URI:
    store_ctx = PostgresStore.from_conn_string(DB_URI)
    store = store_ctx.__enter__()
    store.setup()
    print("Store setup complete.")

gcs_backend = GCSObjectBackend(
    bucket_name=BUCKET_NAME,
    endpoint_url="https://storage.googleapis.com", # The GCS S3-compatible endpoint
    aws_access_key_id=GCS_ACCESS_KEY,
    aws_secret_access_key=GCS_SECRET_KEY
)


def production_backend_factory(rt):
    """
    Factory to inject user-specific context into backends.
    'rt' is the ToolRuntime providing access to the config.
    """
    # Extract user_id from the configurable metadata passed during invocation
    user_info = rt.config.get("configurable", {})
    user_id = user_info.get("user_id", "anonymous_user")
    
    return CompositeBackend(
        # Ephemeral scratchpad (isolated by thread_id automatically)
        default=StateBackend(rt),
        
        routes={
            # Scoped to the user in Postgres for cross-session memory
            "/context/": StoreBackend(rt), 
            "/memories/": StoreBackend(rt),
            # Scoped to a specific folder in GCS: bucket/{database}/users/{user_id}/results/
            "/results/": GCSObjectBackend(
                bucket_name=BUCKET_NAME,
                prefix=f"professor-research-agent/users/{user_id}/results/" 
            )
        }
    )


Store setup complete.


In [7]:
import os
from typing import Literal
from tavily import TavilyClient
from langchain_core.tools import tool

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool
def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

@tool
def read_file(file_path: str):
    """Read a file from the filesystem."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        return f"Error reading file: {e}"

@tool
def write_file(file_path: str, content: str):
    """Write content to a file. Overwrites if exists."""
    try:
        # Ensure directory exists
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(content)
        return f"Successfully wrote to {file_path}"
    except Exception as e:
        return f"Error writing file: {e}"



In [8]:
import os
from deepagents import create_deep_agent, CompiledSubAgent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


model_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0)


# 2. Define the Specialized Subagent Team
subagents = [
    {
        "name": "university-shortlister",
        "description": "Shortlists top-tier universities  based on IT and AI and QS rankings.",
        "system_prompt": "Search for universities within the preferred country  with strong research in AI and IT. Identify specific labs for Applied AI and Quantum Computing. Save your findings to /results/shortlist.md.",
        "tools": [internet_search],
        "model": model_gemini
    },
    {
        "name": "professor-discovery",
        "description": "Finds specific PIs in each shortlisted university working on stateful agents or QML.",
        "system_prompt": "Scan faculty directories to find professors. Filter for keywords: 'stateful agents', 'LLM memory', 'Quantum ML'. Save profiles to /results/profs_raw.json.",
        "tools": [internet_search],
        "model": model_gemini
    },
    {
        "name": "professor-validator",
        "description": "Deep-dives into publications and grant databases to check funding and PhD  openings using the tools provided and internet search.",
        "system_prompt": "Check for active grants (SFI/ERC) and recent publications (2024-2026). Verify if they are currently accepting PhD students. Highlight those matching a B.Tech IT background.",
        "tools": [internet_search],
        "model": model_gemini
    }
]

# 3. Create the Main Orchestrator Agent
agent = create_deep_agent(
    model=model_gemini,
    name="academic-research-lead",
    system_prompt="""
You are the Lead Academic Coordinator. 

Your goal is to coordinate your subagents to find professors and compile a final report.

REPORT INSTRUCTIONS:
As the research progresses, compile the findings into a file named `/results/academic_report.md`.
You MUST use the following table format for the output:

| University Name | Professor Name | Designation | Email | Research Area | Active Grants | Recent Publication | PhD Openings |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |

Use the `write_file` tool to save this report to your persistent GCS storage.
""",
    subagents=subagents
)

In [9]:
import os
from dotenv import load_dotenv

# Load environment variables first
load_dotenv()

from deepagents import create_deep_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from utils.tools import internet_search, read_file, write_file
from utils.backends import production_backend_factory
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
# 1. Define Models
model_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0)

# 2. Define the Specialized Subagent Team
subagents = [
    {
        "name": "university-shortlister",
        "description": "Shortlists top-tier universities based on IT and AI and QS rankings.",
        "system_prompt": "Search for universities within the preferred country with strong research in AI and IT. Identify specific labs for Applied AI and Quantum Computing. Save your findings to /results/shortlist.md.",
        "tools": [internet_search, write_file],
        "model": model_gemini
    },
    {
        "name": "professor-discovery",
        "description": "Finds specific PIs in each shortlisted university working on stateful agents or QML.",
        "system_prompt": "Scan faculty directories to find professors. Filter for keywords: 'stateful agents', 'LLM memory', 'Quantum ML'. Save profiles to /results/profs_raw.json.",
        "tools": [internet_search, write_file, read_file],
        "model": model_gemini
    },
    {
        "name": "professor-validator",
        "description": "Deep-dives into publications and grant databases to check funding and PhD openings.",
        "system_prompt": "Check for active grants (SFI/ERC) and recent publications (2024-2026). Verify if they are currently accepting PhD students. Highlight those matching a B.Tech IT background.",
        "tools": [internet_search, read_file],
        "model": model_gemini
    }
]

# 3. Create the Main Orchestrator Agent
agent = create_deep_agent(
    model=model_gemini,
    name="academic-research-lead",
    system_prompt="""
You are the Lead Academic Coordinator. 

Your goal is to coordinate your subagents to find professors and compile a final report.

REPORT INSTRUCTIONS:
As the research progresses, compile the findings into a file named `/results/academic_report.md`.
You MUST use the following table format for the output:

| University Name | Professor Name | Designation | Email | Research Area | Active Grants | Recent Publication | PhD Openings |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |

Use the `write_file` tool to save this report to your persistent GCS storage (mapped to /results/).
""",
    subagents=subagents,
    tools=[write_file, read_file], # Orchestrator needs to write the final report and read subagent outputs
    backend=production_backend_factory
)




In [11]:
result = agent.invoke({"messages": [{"role": "user", "content": "Find top universities and professors in AI accepting PhD students in the country Australia and my preferred city is Melbourne."}]},)


AttributeError: 'Runtime' object has no attribute 'config'

In [1]:
import pandas as pd 
df = pd.read_csv("result.csv")

In [2]:
df

,University Name,Professor Name,Designation,Email,Research Area,Active Grants,Recent Publication,PhD Openings
0,University of Edinburgh,Sotirios A. Tsaftaris,Chair (Full Professor) in Machine Learning and...,s.tsaftaris@ed.ac.uk,"Machine Learning, Computer Vision, Medical Ima...",Royal Academy of Engineering and Canon Medical...,Self-supervised learning for medical image ana...,No
1,University of Edinburgh,Dr. Steven McDonagh,Senior Lecturer in AI and Computer Vision for ...,s.mcdonagh@ed.ac.uk,"AI, Computer Vision",Not Found,Advances in Neural Information Processing Syst...,Yes
2,University of Edinburgh,Malihe Javidi,PhD Research fellow,Not Found,"Computer Vision, Machine Learning, Medical Ima...",Not Found,SCONe: a community-acquired retinal image repo...,No
3,University of Glasgow,Dr. Marwa Mahmoud,Senior Lecturer (Associate Professor),Marwa.Mahmoud@glasgow.ac.uk,"Computer Vision, Machine Learning, Multimodal ...","R4N (Respect for Neurodevelopment) Grant, SULS...",Interpretable Concept-based Deep Learning Fram...,Yes
4,University of Glasgow,Dr. Fani Deligianni,Senior Lecturer,Fani.Deligianni@glasgow.ac.uk,"Trustworthy AI, Privacy-Preserved Human Motion...",EPSRC Grant: Privacy-Preserved Human Motion An...,Not Found,Yes
5,University of Glasgow,Dr. Gerardo Aragon Camarasa,Senior Lecturer,Gerardo.Aragon-Camarasa@glasgow.ac.uk,"Computer Vision, Robotics, Autonomous Systems,...",EPSRC Capital Equipment grant,Multiway-adapter: Adapting Multimodal Large La...,Yes
6,University of Glasgow,Dr. Paul Siebert,Senior Lecturer,Paul.Siebert@glasgow.ac.uk,"Computer Vision, 3D Vision Systems, Robotics",Not Found,Not Found,Yes
7,University of Glasgow,Dr. Sebastian Stein,Senior Lecturer,Sebastian.Stein@glasgow.ac.uk,"Brain-Computer Interfaces (BCI), Neuromodulati...",ERC Advanced Grant: Designing Interaction Free...,Towards Interaction Design with Active Inferen...,Yes
8,University of Aberdeen,Professor Georgios Leontidis,"Chair in Machine Learning, Interdisciplinary D...",georgios.leontidis@abdn.ac.uk,"Machine Learning, Computer Vision, Data Science",UKRI AI Centre for Doctoral Training in Sustai...,Object-centric learning with capsule networks:...,Yes
9,University of Aberdeen,Dr. Tryphon Lambrou,Senior Lecturer in Computing Science,t.lambrou@abdn.ac.uk,"Machine Learning, Medical Imaging, Fingerprints",Not Found,Not Found,Yes
